In [ ]:
# You have some research articles. These (published) articles are written by your reviewers.
# There is a new article.

!pip install tika

In [ ]:
!pip install pdfminer.six


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 111.0 MB/s eta 0:00:00


In [ ]:
from pdfminer.high_level import extract_text

# Extract text from a PDF file
text0 = extract_text("/content/291.pdf")
text1 = extract_text("/content/292.pdf")

# os.walk('path')

texts = [text0, text1]

In [ ]:
texts[0] # 1st article

'Proceedings of the Twenty-Fifth International Joint Conference on Artificial Intelligence (IJCAI-16)\n\nSupervised Heterogeneous Domain Adaptation via Random Forests\n\nSanatan Sukhija1, Narayanan C Krishnan1, Gurkanwal Singh2\n1Department of Computer Science and Engineering, Indian Institute of Technology Ropar,\n\n⇤\n\nPunjab, India\n\nsanatan@iitrpr.ac.in, ckn@iitrpr.ac.in\n\n2Department of Computer Science and Engineering, PEC University of Technology,\n\nChandigarh, India\n\ngurkanwal.singh7@gmail.com\n\nAbstract\n\nHeterogeneity of features and lack of correspon-\ndence between data points of different domains are\nthe two primary challenges while performing fea-\nture transfer. In this paper, we present a novel su-\npervised domain adaptation algorithm (SHDA-RF)\nthat learns the mapping between heterogeneous\nfeatures of different dimensions. Our algorithm\nuses the shared label distributions present across\nthe domains as pivots for learning a sparse fea-\nture transformation.

In [ ]:
texts[1] # 2nd article --> this was one of the articles that was submitted to your conference

'Artiﬁcial Intelligence 268 (2019) 30–53\n\nContents lists available at ScienceDirect\n\nArtiﬁcial  Intelligence\n\nwww.elsevier.com/locate/artint\n\nSupervised  heterogeneous  feature  transfer  via  random  forests\n\nSanatan Sukhija\n\n∗\n\n,  Narayanan  C. Krishnan\n\nDepartment of Computer Science and Engineering, Indian Institute of Technology Ropar, Rupnagar, Punjab, PB 140001, India\n\na  r  t  i  c  l  e \n\ni  n  f  o\n\na  b  s  t  r  a  c  t\n\nArticle history:\nReceived 18 January 2017\nReceived in revised form 17 August 2018\nAccepted 20 November 2018\nAvailable online 22 November 2018\n\nKeywords:\nFeature transfer learning\nHeterogeneous domain adaptation\nRandom forests\n\nTransfer learning across heterogeneous feature spaces can, in general, be a very diﬃcult \nproblem  in  practice  due  to  the  heterogeneity  of  features  and  lack  of  correspondence \nbetween  data  points  of  different  domains.  In  this  paper,  we  present  a  novel  supervised \ndomain ada

In [ ]:
# Compare two texts

# [Dense] Semantic Embeddings (Doc2Vec), Contextual Embeddings (SciBERT, SentenceBERT, <cls> embeddings from BERT or ots extensions)
# [Lexical] Jaccard Similarity, Bag of Words [TF-IDF], BM25

!pip install transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load SciBERT
tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
model = AutoModel.from_pretrained("allenai/scibert_scivocab_uncased")

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load SciBERT
tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
model = AutoModel.from_pretrained("allenai/scibert_scivocab_uncased")

embeddings = []
for each_text in texts:
  # Tokenize and get embeddings
  inputs = tokenizer(each_text[:500], return_tensors='pt', truncation=True, padding=True)
  with torch.no_grad():
      outputs = model(**inputs)

  # Mean pooling for sentence embedding
  token_embeddings = outputs.last_hidden_state
  attention_mask = inputs['attention_mask']
  mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
  sum_embeddings = torch.sum(token_embeddings * mask_expanded, 1)
  sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
  sentence_embedding = sum_embeddings / sum_mask
  embeddings.append(sentence_embedding)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [ ]:
len(list(embeddings[0])[0])

768

In [ ]:
len(list(embeddings[1])[0])

768

In [ ]:
from sklearn.metrics.pairwise import cosine_distances
1 - cosine_distances(embeddings[0],embeddings[1])[0] # Cosine dissimilarity(distance) -->  [1 - cosine similarity]

array([0.7955539], dtype=float32)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tv = TfidfVectorizer()
tv.fit_transform(texts).toarray().shape

(2, 2511)

In [ ]:
# Code with chunking
from transformers import AutoTokenizer, AutoModel
import torch

# Load SciBERT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
model = AutoModel.from_pretrained("allenai/scibert_scivocab_uncased")

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Your list of text chunks
# Example: texts = ["First document text...", "Second document text...", ...]
embeddings = []

for i, text in enumerate(texts):
    try:
        # Tokenize safely with truncation to 512 tokens
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,        # Ensures no sequence exceeds model limit
            max_length=512,         # BERT models have a hard cap at 512
            padding="max_length"    # Ensures uniform input shape
        )

        # Move tensors to device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Disable gradient computation (inference mode)
        with torch.no_grad():
            outputs = model(**inputs)

        # Mean pooling over the token embeddings (excluding padding tokens)
        attention_mask = inputs['attention_mask']
        last_hidden_state = outputs.last_hidden_state

        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        chunk_embedding = sum_embeddings / sum_mask

        embeddings.append(chunk_embedding.cpu())
        print(f"Processed chunk {i+1}/{len(texts)} successfully.")

    except Exception as e:
        print(f"⚠️ Skipping chunk {i+1} due to error: {e}")
        continue

# Stack all embeddings
embeddings = torch.cat(embeddings, dim=0)
print("✅ Shape of all embeddings:", embeddings.shape)

Processed chunk 1/2 successfully.
Processed chunk 2/2 successfully.
✅ Shape of all embeddings: torch.Size([2, 768])
